In [34]:
import importlib
import subprocess
import sys

required = {
    "langchain": "langchain",
    "langchain_community": "langchain-community",
    "langchain_openai": "langchain-openai",
    "langchain_chroma": "langchain-chroma",
    "langchain_text_splitters": "langchain-text-splitters",
    "pypdf": "pypdf",
    "langchain_nvidia_ai_endpoints": "langchain_nvidia_ai_endpoints",
    "langchainhub": "langchainhub",
}

for module_name, pip_name in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"설치 중: {pip_name}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-U", pip_name], check=True)

print("모든 패키지 준비 완료")

모든 패키지 준비 완료


In [35]:
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## PDF Loader - pages 변수 선언 완료

In [36]:
import requests
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor
from config import get_github_api_url

load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

api_url = get_github_api_url()

response = requests.get(api_url, headers=headers)
files = response.json()

pages = []

def load_pdf(file):
    print("처리 중:", file["name"])
    pdf_url = file["download_url"]
    loader = PyPDFLoader(pdf_url)
    pdf_pages = loader.load_and_split()
    for page in pdf_pages:
        page.metadata["source_file"] = file["name"]
    return pdf_pages

pdf_files = [f for f in files if f["name"].lower().endswith(".pdf")]

with ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(load_pdf, pdf_files)
    for result in results:
        pages.extend(result)

print("총 페이지:", len(pages))

처리 중: 2023년도 사전정보공표 리스트_230701(게시용).pdf
처리 중: 2023년도 사전정보공표 항목 및 담당부서(게시용).pdf
처리 중: 2024년도 사전정보공표 항목 및 담당부서(게시용).pdf
총 페이지: 12


## Chunk 분할

In [37]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(pages)

## Chunk들 임베딩으로 저장

In [38]:
import requests
from langchain_core.embeddings import Embeddings
NVIDIA_API_KEY = os.getenv("NVIDIA_BUILD_KEY")

class NVIDIAEmbeddingsCustom(Embeddings):

    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://integrate.api.nvidia.com/v1/embeddings"
        self.model = "nvidia/nemotron-3-embed-1b"

    def embed_documents(self, texts):
        payload = {
            "model": self.model,
            "encoding_format": "float",
            "truncate": "NONE",
            "input": texts
        }

        headers = {
            "accept": "application/json",
            "content-type": "application/json",
            "authorization": f"Bearer {self.api_key}"
        }

        response = requests.post(
            self.url,
            json=payload,
            headers=headers
        )

        response.raise_for_status()

        return [
            item["embedding"]
            for item in response.json()["data"]
        ]

    def embed_query(self, text):
        payload = {
            "model": self.model,
            "encoding_format": "float",
            "truncate": "NONE",
            "input": [text]
        }

        headers = {
            "accept": "application/json",
            "content-type": "application/json",
            "authorization": f"Bearer {self.api_key}"
        }

        response = requests.post(
            self.url,
            json=payload,
            headers=headers
        )

        response.raise_for_status()

        return response.json()["data"][0]["embedding"]


In [39]:
embeddings = NVIDIAEmbeddingsCustom(NVIDIA_API_KEY)

vectorstore = Chroma.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

## LLM 선언

In [40]:
from langsmith import Client

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="google/diffusiongemma-26b-a4b-it",
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
    temperature=0,
    max_tokens=100,
    timeout=180,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

#Retriever로 검색한 유사 문서의 내용을 하나의 string으로 결합
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Chain 선언

In [41]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## RAG 사용

In [42]:
user_prompt = "첨부한 파일에 대해서 설명해줘. 혹시 7월에는 공표가 안되었나?"

In [43]:
for chunk in rag_chain.stream(user_prompt):
    print(chunk, end="", flush=True)

첨부된 내용은 재무·회계, 정보화·보안, 계약·구매, 기술·교육 등 다양한 분야의 업무 현황과 공표 주기를 설명하는 목록입니다. 7월에는 보유도서 목록 현황, 보안점검 결과, 전력수급계획 대상사업, 500만원 이상 수의계약 현황, 1억원 이상 발주계획 등이 공표되는 것으로 확인됩니다.